# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from train import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 478

In [3]:
df = pd.read_csv(f"../../data/top30groups/anonLoc/combined/combined{partition}.csv")

In [4]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [5]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Weapon type prediction

In [6]:
torch.cuda.empty_cache()


In [7]:
label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds, y_trues, logs = [], [], []
from itertools import product
import os
import random
# Hyperparameter grid
#Config: {'lr': 0.01, 'n_tree': 100, 'tree_depth': 9, 'tree_feature_rate': 0.1, 'feat_dropout': 0.2, 'out_size_nrf': 512, 'partition': 'gtd478', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}

param_grid = {
    'lr': [0.01],
    'n_tree': [100, 120, 150],
    'tree_depth': [7,8,9,10],
    'tree_feature_rate': [0.1, 0.3, 0.5],
    'feat_dropout': [0.1, 0.2],
    'out_size_nrf': [512]
    }

# Convert to list of dicts (cartesian product)
grid_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())
sampled_combos = random.sample(grid_combos, 50)

for col in continuous_cols:
    print(f"\nTraining model for {col} prediction...")
    best_run = None
    best_score = -1

    for combo in sampled_combos:
        args = {
            **dict(zip(param_names, combo)),
            'partition': f"gtd{partition}",
            'n_class': len(label_index),
            'epochs': 500,
            'final_evaluation': False
        }

        print(f"Running config: {args}")
        acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs =train_joint(df, args, label_index, verbose=True)

        if acc > best_score:
            best_score = acc
            best_run = {
                "args": args,
                "acc": acc,
                "epoch": epoch,
                "y_pred": y_pred_decoded,
                "y_true": y_true_decoded,
                "precision": p,
                "recall": r,
                "f1": f1,
                "micro": (p_micro, r_micro, f1_micro),
                "macro": (p_macro, r_macro, f1_macro),
                "auroc": (auc_w, auc_mi, auc_ma),
                "epoch_logs": epoch_logs
            }


    # Save best results
    if best_run:
            print(f"\nRetraining best config for {col} on partition {partition} with 3000 epochs...")
            best_args_final = best_run["args"].copy()
            best_args_final["epochs"] = 3000
            best_args_final["final_evaluation"] = True

            acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, \
            p_micro, r_micro, f1_micro, \
            p_macro, r_macro, f1_macro, \
            auc_w, auc_mi, auc_ma, epoch_logs = train_joint(df, best_args_final, label_index, verbose=True)

            # overwrite with retrained results
            best_run.update({
                "args": best_args_final,
                "acc": acc,
                "epoch": epoch,
                "y_pred": y_pred_decoded,
                "y_true": y_true_decoded,
                "precision": p,
                "recall": r,
                "f1": f1,
                "micro": (p_micro, r_micro, f1_micro),
                "macro": (p_macro, r_macro, f1_macro),
                "auroc": (auc_w, auc_mi, auc_ma),
                "epoch_logs": epoch_logs
            })

            # Save results
            os.makedirs(f"Results{partition}", exist_ok=True)
            results_path = f"Results{partition}/Results_{col}_prediction"
            with open(results_path, "w") as f:
                f.write(f"Best acc: {best_run['acc']:.4f} at epoch {best_run['epoch']} for {col} prediction\n")
                f.write(f"Config: {best_run['args']}\n")
                f.write(f"Weighted Precision: {best_run['precision']:.4f}, Recall: {best_run['recall']:.4f}, F1: {best_run['f1']:.4f}\n")
                f.write(f"Macro Precision: {best_run['macro'][0]:.4f}, Recall: {best_run['macro'][1]:.4f}, F1: {best_run['macro'][2]:.4f}\n")
                f.write(f"Micro Precision: {best_run['micro'][0]:.4f}, Recall: {best_run['micro'][1]:.4f}, F1: {best_run['micro'][2]:.4f}\n")
                f.write(f"AUROC Weighted: {best_run['auroc'][0]:.4f}, Micro: {best_run['auroc'][1]:.4f}, Macro: {best_run['auroc'][2]:.4f}\n")

            log_path = f"Results{partition}/epoch_logs_{col}_prediction"
            with open(log_path, "w") as f:
                f.write('\n'.join(f"{x:.4f}" for x in best_run['epoch_logs']))

            y_preds.append(best_run['y_pred'])
            y_trues.append(best_run['y_true'])

    print(f"Best score for partition {partition}: {best_score:.4f}")


Training model for weaptype1 prediction...
Running config: {'lr': 0.01, 'n_tree': 150, 'tree_depth': 10, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'out_size_nrf': 512, 'partition': 'gtd478', 'n_class': 30, 'epochs': 500, 'final_evaluation': False}
Epoch 000 | NRF Loss: 3.4012 | Val Acc: 0.1653
Epoch 050 | NRF Loss: 2.7415 | Val Acc: 0.3927
Epoch 100 | NRF Loss: 2.2036 | Val Acc: 0.4684
Epoch 150 | NRF Loss: 1.8404 | Val Acc: 0.4990
Epoch 200 | NRF Loss: 1.6083 | Val Acc: 0.5163
Epoch 250 | NRF Loss: 1.4536 | Val Acc: 0.5306
Epoch 300 | NRF Loss: 1.3494 | Val Acc: 0.5437
Epoch 350 | NRF Loss: 1.2755 | Val Acc: 0.5483
Epoch 400 | NRF Loss: 1.2202 | Val Acc: 0.5538
Epoch 450 | NRF Loss: 1.1776 | Val Acc: 0.5576
Best validation acc: 0.5608 @ epoch 486
Running config: {'lr': 0.01, 'n_tree': 100, 'tree_depth': 7, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'out_size_nrf': 512, 'partition': 'gtd478', 'n_class': 30, 'epochs': 500, 'final_evaluation': False}
Epoch 000 | NRF Loss: 3.401

In [8]:
#test 0.9355495572090149
#Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
#Early stopping at epoch 779
#Best validation acc: 0.9331 @ epoch 679
#{'args': {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.9355495572090149,

In [9]:
print(best_run)

{'args': {'lr': 0.01, 'n_tree': 100, 'tree_depth': 9, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'out_size_nrf': 512, 'partition': 'gtd478', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.5673611164093018, 'epoch': 1627, 'y_pred': ['Shining Path (SL)', 'Boko Haram', 'Tehrik-i-Taliban Pakistan (TTP)', 'Liberation Tigers of Tamil Eelam (LTTE)', 'Muslim extremists', 'Communist Party of India - Maoist (CPI-Maoist)', 'Revolutionary Armed Forces of Colombia (FARC)', 'African National Congress (South Africa)', "Donetsk People's Republic", 'African National Congress (South Africa)', 'Islamic State of Iraq and the Levant (ISIL)', "Kurdistan Workers' Party (PKK)", 'Liberation Tigers of Tamil Eelam (LTTE)', 'Al-Shabaab', 'Muslim extremists', 'Nicaraguan Democratic Force (FDN)', "Donetsk People's Republic", 'Shining Path (SL)', 'Farabundo Marti National Liberation Front (FMLN)', 'Taliban', 'Fulani extremists', 'Communist Party of India - Maoist (CPI-Maoist)', 'African Natio

In [10]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': True
}
0.9287652969360352

"""

'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1000,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': len(label_index),\n    \'final_evaluation\': True\n}\n0.9287652969360352\n\n'

In [11]:
best_acc

NameError: name 'best_acc' is not defined

In [ ]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [ ]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])